In [1]:
import geopandas as gpd
from scipy.spatial import cKDTree
import numpy as np
import pandas as pd

## Find Duplicates

In [2]:
DISTANCE_THRESHOLD = 10
TIME_THRESHOLD = 180

In [3]:
# get geodata frame in meters for montreal epsg=32188
potholes = gpd.read_file('datasets/potholes_cleaned.gpkg')
potholes_proj = potholes.to_crs(epsg=32188)

In [4]:
# sort by date
potholes_proj = potholes_proj.sort_values('Date').reset_index(drop=True)
# get coords
coords = np.array([(geom.x, geom.y) for geom in potholes_proj.geometry])
# build spatial index
tree = cKDTree(coords)
# find pairs/duplicates
pairs = tree.query_pairs(r=DISTANCE_THRESHOLD)

print (f"Found {len(pairs)} pairs within {DISTANCE_THRESHOLD}m")

Found 26391632 pairs within 10m


Found 26391632 pairs within 10m

In [5]:
MIN_DAYS = 7
MAX_DAYS = 180

# Get dates as array for fast lookup
dates = potholes_proj['Date'].values

repeat_indices = set()

for i, j in pairs:
    days_diff = abs((dates[j] - dates[i]) / np.timedelta64(1, 'D'))

    if MIN_DAYS <= days_diff <= MAX_DAYS:
        # Mark the earlier repair as one that will need re-repair
        earlier = i if dates[i] < dates[j] else j
        repeat_indices.add(earlier)

print(f"Found {len(repeat_indices)} repairs that were re-repaired within {MAX_DAYS} days")

Found 514535 repairs that were re-repaired within 180 days


Found 514535 repairs that were re-repaired within 180 days.

## Create target variable

In [6]:
# Create target column (0 = no repeat, 1 = repeat within 180 days)
potholes_proj['repeat'] = 0
potholes_proj.loc[list(repeat_indices), 'repeat'] = 1

print(potholes_proj['repeat'].value_counts())
print(f"\nRepeat rate: {potholes_proj['repeat'].mean():.1%}")

repeat
1    514535
0    512732
Name: count, dtype: int64

Repeat rate: 50.1%


In [7]:
# Convert back to WGS84 for consistency with other datasets
potholes_target = potholes_proj.to_crs("EPSG:4326")

# Save
potholes_target.to_file("datasets/potholes_with_target.gpkg", driver="GPKG")

print(f"Saved {len(potholes_target)} repairs with target variable")

Saved 1027267 repairs with target variable
